In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import io
import base64
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# fallback chain so the Hindi (Devanagari) and Tamil legend labels render alongside Latin text
plt.rcParams['font.family'] = ['Noto Sans', 'Noto Sans Devanagari', 'Noto Sans Tamil']

In [2]:
apps_df = pd.read_csv('Play Store Data.csv')

In [3]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [6]:
apps_df = apps_df.drop_duplicates()
required_cols = ['App', 'Category', 'Reviews', 'Installs', 'Last Updated']
apps_df = apps_df.dropna(subset=required_cols)
print(f"Rows after removing duplicates and nulls: {len(apps_df)}")

Rows after removing duplicates and nulls: 10358


In [7]:
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df = apps_df.dropna(subset=['Reviews'])
apps_df['Reviews'] = apps_df['Reviews'].astype(int)
print(f"Rows after Reviews cleaning: {len(apps_df)}")

Rows after Reviews cleaning: 10357


In [8]:
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')
apps_df = apps_df.dropna(subset=['Installs'])
apps_df['Installs'] = apps_df['Installs'].astype(int)
print(f"Rows after Installs cleaning: {len(apps_df)}")

Rows after Installs cleaning: 10357


In [9]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df = apps_df.dropna(subset=['Last Updated'])

# month-year column used later for grouping and the x-axis
apps_df['Month_Year'] = apps_df['Last Updated'].dt.to_period('M')
print(f"Rows after date cleaning: {len(apps_df)}")

Rows after date cleaning: 10357


In [10]:
apps_df = apps_df[apps_df['Reviews'] > 500]
print(f"After Reviews filter (>500): {len(apps_df)}")

After Reviews filter (>500): 5958


In [11]:
# app name should not start with X, Y or Z
apps_df = apps_df[~apps_df['App'].str[0].str.upper().isin(['X', 'Y', 'Z'])]
print(f"After App name filter (no X/Y/Z start): {len(apps_df)}")

After App name filter (no X/Y/Z start): 5804


In [12]:
# app name should not contain the letter S anywhere
apps_df = apps_df[~apps_df['App'].str.upper().str.contains('S', na=False)]
print(f"After App name filter (no letter S): {len(apps_df)}")

After App name filter (no letter S): 1924


In [13]:
# category must start with E, C or B, or be Dating
apps_df = apps_df[apps_df['Category'].str.startswith(('E', 'C', 'B')) | (apps_df['Category'] == 'DATING')]
print(f"After Category filter (E/C/B or Dating): {len(apps_df)}")
print(apps_df['Category'].unique())

After Category filter (E/C/B or Dating): 252
['BEAUTY' 'BOOKS_AND_REFERENCE' 'BUSINESS' 'COMICS' 'COMMUNICATION'
 'DATING' 'EDUCATION' 'ENTERTAINMENT' 'EVENTS']


In [14]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Month_Year
107,Ulta Beauty,BEAUTY,4.7,42050,Varies with device,1000000,Free,0,Everyone,Beauty,2018-06-05,5.4,5.0 and up,2018-06
136,Rainbow Camera,BEAUTY,3.7,3871,23M,1000000,Free,0,Everyone,Beauty,2018-07-30,2.2.0,4.0 and up,2018-07
140,E-Book Read - Read Book for free,BOOKS_AND_REFERENCE,4.5,1857,4.9M,50000,Free,0,Everyone,Books & Reference,2018-08-03,1.3.2,4.4 and up,2018-08
141,Download free book with green book,BOOKS_AND_REFERENCE,4.6,4478,9.5M,100000,Free,0,Everyone 10+,Books & Reference,2017-07-31,1.1,4.0 and up,2017-07
142,Wikipedia,BOOKS_AND_REFERENCE,4.4,577550,Varies with device,10000000,Free,0,Everyone,Books & Reference,2018-08-02,Varies with device,Varies with device,2018-08


In [15]:
# nice display names for the categories, then translate the 3 required ones
cat_names = {
    'BEAUTY': 'Beauty',
    'BOOKS_AND_REFERENCE': 'Books & Reference',
    'BUSINESS': 'Business',
    'COMICS': 'Comics',
    'COMMUNICATION': 'Communication',
    'DATING': 'Dating',
    'EDUCATION': 'Education',
    'ENTERTAINMENT': 'Entertainment',
    'EVENTS': 'Events'
}

translate = {
    'Beauty': 'सौंदर्य',
    'Business': 'வணிகம்',
    'Dating': 'Partnersuche'
}

legend_names = {}
for cat in cat_names:
    name = cat_names[cat]
    if name in translate:
        name = translate[name]
    legend_names[cat] = name

legend_names

{'BEAUTY': 'सौंदर्य',
 'BOOKS_AND_REFERENCE': 'Books & Reference',
 'BUSINESS': 'வணிகம்',
 'COMICS': 'Comics',
 'COMMUNICATION': 'Communication',
 'DATING': 'Partnersuche',
 'EDUCATION': 'Education',
 'ENTERTAINMENT': 'Entertainment',
 'EVENTS': 'Events'}

In [16]:
grp = apps_df.groupby(['Month_Year', 'Category'])['Installs'].sum().reset_index()
pivot_df = grp.pivot(index='Month_Year', columns='Category', values='Installs')
pivot_df = pivot_df.fillna(0)
pivot_df = pivot_df.sort_index()
pivot_df

Category,BEAUTY,BOOKS_AND_REFERENCE,BUSINESS,COMICS,COMMUNICATION,DATING,EDUCATION,ENTERTAINMENT,EVENTS
Month_Year,,,,,,,,,
2014-01,0.0,0.0,0.0,0.0,1.000000e+05,0.0,0.0,0.0,0.0
2014-03,0.0,0.0,0.0,0.0,1.000000e+05,0.0,0.0,0.0,0.0
2014-05,0.0,0.0,100000.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0
2014-07,0.0,0.0,0.0,0.0,1.000000e+07,0.0,10000.0,0.0,0.0
2014-10,0.0,500000.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0
2014-11,0.0,5000000.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0
2015-01,0.0,0.0,0.0,0.0,0.000000e+00,0.0,100000.0,0.0,0.0
2015-03,0.0,0.0,0.0,0.0,1.000000e+06,0.0,0.0,0.0,0.0
2015-06,0.0,0.0,0.0,0.0,0.000000e+00,0.0,1000000.0,0.0,0.0


In [17]:
# month-over-month growth per category, based on the monthly totals
growth_df = pivot_df.pct_change() * 100
growth_df = growth_df.replace([np.inf, -np.inf], np.nan)
growth_df

Category,BEAUTY,BOOKS_AND_REFERENCE,BUSINESS,COMICS,COMMUNICATION,DATING,EDUCATION,ENTERTAINMENT,EVENTS
Month_Year,,,,,,,,,
2014-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-03,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
2014-05,NaN,NaN,NaN,NaN,-100.000000,NaN,NaN,NaN,NaN
2014-07,NaN,NaN,-100.000000,NaN,NaN,NaN,NaN,NaN,NaN
2014-10,NaN,NaN,NaN,NaN,-100.000000,NaN,-100.000000,NaN,NaN
2014-11,NaN,900.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01,NaN,-100.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-03,NaN,NaN,NaN,NaN,NaN,NaN,-100.000000,NaN,NaN
2015-06,NaN,NaN,NaN,NaN,-100.000000,NaN,NaN,NaN,NaN


In [18]:
ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)

current_decimal = now_ist.hour + now_ist.minute / 60
window_start = 18.0   # 6:00 PM
window_end = 21.0     # 9:00 PM

in_window = window_start <= current_decimal <= window_end
print(f"Current IST: {now_ist.strftime('%I:%M %p')} | Chart visible: {in_window}")

Current IST: 01:13 PM | Chart visible: False


In [19]:
colors = {
    'BEAUTY': '#e377c2',
    'BOOKS_AND_REFERENCE': '#8c564b',
    'BUSINESS': '#1f77b4',
    'COMICS': '#ff7f0e',
    'COMMUNICATION': '#2ca02c',
    'DATING': '#d62728',
    'EDUCATION': '#9467bd',
    'ENTERTAINMENT': '#17becf',
    'EVENTS': '#bcbd22'
}

In [20]:
# chart_base64 holds the chart image so it can be embedded in the dashboard later
chart_base64 = None

if not in_window:
    print("Time-Series Chart can only be viewed between 6 PM and 9 PM IST.")
elif apps_df.empty:
    print("No data available for the selected filtering criteria.")
else:
    cats = list(pivot_df.columns)
    x_pos = list(range(len(pivot_df.index)))
    x_labels = [p.strftime('%b-%Y') for p in pivot_df.index]

    plt.figure(figsize=(14, 7))

    for cat in cats:
        y_vals = pivot_df[cat].values
        plt.plot(x_pos, y_vals, marker='o', markersize=4, linewidth=1.5,
                 color=colors[cat], alpha=0.75, label=legend_names[cat])

        # redraw any segment with more than 20% month-over-month growth as a bold highlighted line
        for i in range(1, len(pivot_df)):
            growth = growth_df[cat].iloc[i]
            if pd.notna(growth) and growth > 20:
                xs = [i - 1, i]
                ys = [y_vals[i - 1], y_vals[i]]
                plt.plot(xs, ys, linewidth=3.5, color=colors[cat], alpha=1.0, zorder=5)
                plt.fill_between(xs, ys, alpha=0.15, color=colors[cat], zorder=1)

    plt.legend(title='Category', loc='upper left')
    plt.xticks(x_pos, x_labels, rotation=45, ha='right')
    plt.xlabel('Month-Year')
    plt.ylabel('Total Installs')
    plt.title("Total Installs Over Time by Category (bold segment = >20% MoM growth)")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # save the chart as base64 text so it can be embedded straight into the dashboard html
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    chart_base64 = base64.b64encode(buf.read()).decode('utf-8')

    plt.show()

Time-Series Chart can only be viewed between 6 PM and 9 PM IST.


In [22]:
# build the html body depending on which case we are in
if chart_base64 is not None:
    body = f'<img src="data:image/png;base64,{chart_base64}" style="max-width:100%;">'
elif apps_df.empty:
    body = '<p class="notice">No data available for the selected filtering criteria.</p>'
else:
    body = f'<p class="notice">Time-Series Chart can only be viewed between 6 PM and 9 PM IST.<br>Current IST time: {now_ist.strftime("%I:%M %p")}</p>'

dashboard_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Play Store Dashboard</title>
<style>
body {{ background: #0d0d0d; color: white; font-family: Arial, sans-serif; text-align: center; padding: 30px; }}
h1 {{ color: #a78bfa; margin-bottom: 5px; }}
.subtitle {{ color: #9ca3af; font-size: 14px; margin-bottom: 30px; }}
.notice {{ color: #f59e0b; font-size: 16px; padding: 40px; }}
</style>
</head>
<body>
<h1>Play Store Analytics Dashboard</h1>
<div class="subtitle">Total Installs Over Time by Category | Reviews &gt; 500 | App name excludes X/Y/Z-start and letter 'S' | Category E/C/B or Dating</div>
{body}
</body>
</html>
"""

with open('dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print("Dashboard saved as dashboard.html")
from IPython.display import IFrame
with open('dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)
print("Dashboard saved as dashboard.html")

IFrame(src='dashboard.html', width='100%', height=600)

Dashboard saved as dashboard.html
Dashboard saved as dashboard.html
